<a href="https://colab.research.google.com/github/zencolab/WhatDreamsCost-ComfyUI/blob/main/MiniMax_H3_ref2va_ComfyUI_Colab_G4_FRP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MiniMax-H3 参考图生视频(带音频) — Colab G4 精简启动脚本

适配工作流：`video_minimax_h3_r2v.json`　运行时：**G4 高 RAM**（RTX PRO 6000 Blackwell / 96GB）

结构与穿透方式完全对齐仓库内 `导演LTX_Director_Workflow_G4.ipynb`：FRP tcp 模式、无 token、`frp_0.56.0_linux_amd64`、Cell 6 写配置 / Cell 8 启动。

**按工作流 JSON 逐节点核对结果**

| 项 | 结论 |
| --- | --- |
| 节点仓库 | **0 个第三方**：23 个节点 `cnr_id` 全部是 `comfy-core`(0.30.0)，只装 ComfyUI-Manager 备用 |
| 主模型 | `minimax_h3_ref2va_pruned_int8_convrot.safetensors`（UNETLoader） |
| 文本编码器 | `qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors`（CLIPLoader，type=minimax，NVFP4 需 Blackwell，G4 原生支持） |
| VAE | 视频 `fp16` + 音频 `fp32` 两个 |
| 模型数量 | 4 个，约 34GB |
| 分辨率 | 保持模板原值 **1344×768 / 5s / 24fps**（96GB 显存无需缩水） |
| 内网穿透 | FRP tcp 无 token（http://usoren.usdream.dpdns.org:8091） |

按顺序执行 Cell 1 → 8。重启界面时只需重跑 Cell 6 与 Cell 8。

> 服务端前提：frps 配置里 **不能**有 `auth.token`（或 `[common]` 下的 `token`），改完 `systemctl restart frps`；端口 8091 需空闲且在 `allowPorts` 范围内。
> MiniMaxH3ReferenceToVideo 是 ComfyUI PR # 合入的新节点，**必须用 master 最新代码**，稳定版 tag 没有。

In [ ]:
# ==========================================
# Cell 1: 基础环境 (ComfyUI 本体)
# 本 Notebook 已严格按 video_minimax_h3_r2v.json 精简
# 目标运行时: Colab "G4 高 RAM" (NVIDIA RTX PRO 6000 Blackwell, 96GB VRAM)
# ==========================================
import os, subprocess
print("=== 开始安装 ComfyUI 本体 ===")
%cd /content

if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI
else:
    !cd ComfyUI && git pull

%cd /content/ComfyUI
# MiniMaxH3ReferenceToVideo 来自 PR #15224, 必须是 master 最新提交
!git checkout master -q && git pull -q
!pip install -q -r requirements.txt huggingface_hub hf_transfer

# --- Blackwell (sm_120) 自检: 确认 torch 没被 requirements.txt 降级成不支持的旧版 ---
import torch
print("torch:", torch.__version__, "| cuda:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| capability:", torch.cuda.get_device_capability(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("VRAM: %.0f GB" % total)
    if torch.cuda.get_device_capability(0)[0] >= 12:
        print("检测到 Blackwell, int8 / nvfp4 权重可原生运行")
else:
    print("未检测到 GPU, 请检查运行时类型")

# --- 通用插件: 工作流本身不需要, 仅 Manager 便于排查缺失节点 ---
def install_node(repo_url):
    folder_name = repo_url.split('/')[-1].replace('.git', '')
    target_path = f"/content/ComfyUI/custom_nodes/{folder_name}"
    if not os.path.exists(target_path):
        !git clone {repo_url} {target_path}
        if os.path.exists(f"{target_path}/requirements.txt"):
            !pip install -q -r {target_path}/requirements.txt
    else:
        !cd {target_path} && git pull

base_nodes = [
    "https://github.com/ltdrdata/ComfyUI-Manager.git",     # 可选, 排错用
]
for node in base_nodes:
    install_node(node)

print("\nCell 1 完成")

In [ ]:
# ==========================================
# Cell 2: 自定义节点安装 —— 本工作流【一个第三方仓库都不需要】
# ------------------------------------------
# 工作流内 23 个节点的归属核对结果 (JSON 里每个节点的 properties.cnr_id 都是 comfy-core):
#   模型加载   : UNETLoader / CLIPLoader / VAELoader x2
#   H3 专用    : MiniMaxH3ReferenceToVideo   <- PR #15224, 核心自带, 需 master
#   采样       : RandomNoise / KSamplerSelect / BasicScheduler / BasicGuider /
#                SamplerCustomAdvanced
#   解码输出   : VAEDecode / VAEDecodeAudio / CreateVideo / SaveVideo
#   辅助       : LoadImage x2 / PrimitiveFloat / PrimitiveStringMultiline /
#                ResolutionSelector / ComfyMathExpression / MarkdownNote x3
# 所以这里只做校验, 不装任何第三方节点。
# ==========================================
import os, subprocess

comfyui_dir = "/content/ComfyUI"
custom_nodes_dir = os.path.join(comfyui_dir, "custom_nodes")
os.makedirs(custom_nodes_dir, exist_ok=True)

node_repos = [
    # ==========================================
    # 以下为常见但本工作流未用到的节点, 已全部注释
    # ==========================================
    # "https://github.com/Lightricks/ComfyUI-LTXVideo.git",
    # "https://github.com/kijai/ComfyUI-KJNodes.git",
    # "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",  # 用核心 CreateVideo/SaveVideo
    # "https://github.com/evanspearman/ComfyMath.git",                # ComfyMathExpression 已进核心
    # "https://github.com/rgthree/rgthree-comfy.git",
]

for repo in node_repos:
    repo_name = repo.split('/')[-1].replace('.git', '')
    repo_path = os.path.join(custom_nodes_dir, repo_name)
    if not os.path.exists(repo_path):
        subprocess.run(["git", "clone", repo, repo_path])
    req = os.path.join(repo_path, "requirements.txt")
    if os.path.exists(req):
        subprocess.run(["pip", "install", "-r", req, "--quiet"])

# ==========================================
# 核心节点存在性校验: 直接在源码里 grep, 不用启动服务
# ==========================================
print("=== 校验核心节点 ===")
hit = subprocess.run(
    "grep -rl 'MiniMaxH3ReferenceToVideo' /content/ComfyUI/comfy_extras /content/ComfyUI/nodes.py",
    shell=True, capture_output=True, text=True).stdout.strip()
if hit:
    print("MiniMaxH3ReferenceToVideo 已存在于:")
    print(hit)
else:
    print("未找到 MiniMaxH3ReferenceToVideo！说明 ComfyUI 代码不够新。")
    print("请回到 Cell 1 确认执行了 git pull (分支 master)。")

# 防止任何 requirements.txt 把 torch 降级 (Blackwell 需要 cu128+ 的 torch)
import torch
print("修复后 torch:", torch.__version__, "| cuda:", torch.version.cuda)
print("\n节点检查完成 (无需安装第三方仓库)")

In [ ]:
# ==========================================
# Cell 3: 额外 TTS / 音频模型 —— 本工作流完全未使用, 整段已注释
# ------------------------------------------
# 原因: MiniMax-H3 的音轨由模型自身生成, 经 VAEDecodeAudio +
#       minimax_h3_audio_vae_fp32 直接解码, 不需要任何外挂 TTS。
# ==========================================

# from huggingface_hub import snapshot_download
# snapshot_download(repo_id="drbaph/s2-pro-fp8",
#                   local_dir="/content/ComfyUI/models/FishAudioS2/s2-pro-fp8")

print("已跳过外挂 TTS 模型 (本工作流不需要)")

In [ ]:
# ==========================================
# Cell 4: 模型下载 (严格按工作流内 Loader 节点的 widget 值核对)
# ------------------------------------------
# 工作流实际引用的文件名 (从 JSON 里逐个读出):
#   UNETLoader  (id 127) -> minimax_h3_ref2va_pruned_int8_convrot.safetensors  ~14GB
#   CLIPLoader  (id 128) -> qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors       ~19GB  type=minimax
#   VAELoader   (id 119) -> minimax_h3_video_vae_fp16.safetensors
#   VAELoader   (id 120) -> minimax_h3_audio_vae_fp32.safetensors
# 共 4 个文件, 约 34GB。工作流里没有 CheckpointLoaderSimple, 也没有任何 LoraLoader。
# ==========================================
import os
import shutil
from huggingface_hub import hf_hub_download
from concurrent.futures import ThreadPoolExecutor

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except Exception:
    print("未读到 HF_TOKEN (左侧 Secrets), 公开仓库仍可下载, 受限仓库会失败")

UNET_DIR = "/content/ComfyUI/models/diffusion_models"
REPO = "Comfy-Org/MiniMax-H3"

downloads = [
    # --- 1. 主模型: ref2va INT8 (UNETLoader, weight_dtype=default) ---
    {"repo_id": REPO,
     "filename": "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",
     "final_dir": UNET_DIR, "flatten": True},

    # --- 2. 文本编码器: Qwen3-VL-32B NVFP4 AWQ (CLIPLoader, type=minimax) ---
    #     NVFP4 需要 Blackwell, 正好匹配 G4 运行时
    {"repo_id": REPO,
     "filename": "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
     "final_dir": "/content/ComfyUI/models/text_encoders", "flatten": True},

    # --- 3. 视频 / 音频 VAE (VAELoader x2) ---
    {"repo_id": REPO, "filename": "vae/minimax_h3_video_vae_fp16.safetensors",
     "final_dir": "/content/ComfyUI/models/vae", "flatten": True},
    {"repo_id": REPO, "filename": "vae/minimax_h3_audio_vae_fp32.safetensors",
     "final_dir": "/content/ComfyUI/models/vae", "flatten": True},

    # ==========================================
    # 以下为其它精度版本, 本工作流未引用, 已注释 (需要时取消注释即可)
    # ==========================================
    # {"repo_id": REPO, "filename": "diffusion_models/minimax_h3_ref2va_bf16.safetensors", "final_dir": UNET_DIR, "flatten": True},
    # {"repo_id": REPO, "filename": "text_encoders/qwen3vl_32b_minimax_h3_fp8_scaled.safetensors", "final_dir": "/content/ComfyUI/models/text_encoders", "flatten": True},
]


def download_model(task):
    try:
        os.makedirs(task["final_dir"], exist_ok=True)
        base_name = task.get("rename_to", os.path.basename(task["filename"]))
        final_path = os.path.join(task["final_dir"], base_name)

        if os.path.exists(final_path):
            print("已存在, 跳过下载: " + base_name)
            return

        downloaded_path = hf_hub_download(
            repo_id=task["repo_id"],
            filename=task["filename"],
            local_dir=task["final_dir"],
            repo_type="model",
        )

        if ("flatten" in task or "rename_to" in task) and downloaded_path != final_path:
            os.makedirs(os.path.dirname(final_path), exist_ok=True)
            if os.path.exists(final_path):
                os.remove(final_path)
            shutil.move(downloaded_path, final_path)

        print("成功就绪: " + base_name)
    except Exception as e:
        print("失败 %s: %s" % (task['filename'].split('/')[-1], e))


print("开始并发下载本工作流所需的 %d 个模型 (约 34GB)..." % len(downloads))
with ThreadPoolExecutor(max_workers=4) as executor:
    list(executor.map(download_model, downloads))

# 兼容旧版 ComfyUI: 在 models/unet 下做一份硬链接
legacy_dir = "/content/ComfyUI/models/unet"
os.makedirs(legacy_dir, exist_ok=True)
for f in os.listdir(UNET_DIR):
    src, dst = os.path.join(UNET_DIR, f), os.path.join(legacy_dir, f)
    if not os.path.exists(dst):
        try:
            os.link(src, dst)
        except OSError:
            pass

# 清理 hf_hub 的中间目录
for d in [UNET_DIR, "/content/ComfyUI/models/text_encoders", "/content/ComfyUI/models/vae"]:
    for sub in ["diffusion_models", "text_encoders", "vae", ".cache"]:
        p = os.path.join(d, sub)
        if os.path.isdir(p):
            shutil.rmtree(p, ignore_errors=True)

import subprocess
subprocess.run(["du", "-sh", "/content/ComfyUI/models"])
print("\n模型下载完毕")

In [ ]:
# ==========================================
# Cell 5: 工作流 JSON + 参考图
# ------------------------------------------
# 工作流来自 ComfyUI 官方模板库 Comfy-Org/workflow_templates
# 两张参考图 (red_superboy_on_city_roof.png / mecha_dragon_lightning.png)
# 不在公开 input 清单里, 需要手动上传, 或在界面里换成你自己的图。
# ==========================================
import os
import urllib.request

wf_dir = "/content/ComfyUI/user/default/workflows"
in_dir = "/content/ComfyUI/input"
os.makedirs(wf_dir, exist_ok=True)
os.makedirs(in_dir, exist_ok=True)

wf_url = ("https://raw.githubusercontent.com/Comfy-Org/workflow_templates/"
          "main/templates/video_minimax_h3_r2v.json")
wf_path = os.path.join(wf_dir, "video_minimax_h3_r2v.json")

try:
    urllib.request.urlretrieve(wf_url, wf_path)
    print("工作流已就绪: " + wf_path)
    print("启动 ComfyUI 后在左侧 Workflows 面板直接打开")
except Exception as e:
    print("下载失败: %s" % e)
    print("可在界面里手动拖入本地 video_minimax_h3_r2v.json")

# --- 上传两张参考图 (可跳过, 之后在界面里选图也行) ---
try:
    from google.colab import files
    print("\n请选择参考图 (可多选; 不需要就直接取消):")
    up = files.upload()
    for name, data in up.items():
        with open(os.path.join(in_dir, name), "wb") as f:
            f.write(data)
        print("已放入 input/: " + name)
except Exception as e:
    print("跳过上传: %s" % e)

print("\nCell 5 完成")

In [ ]:
# ==========================================
# Cell 6: FRP 内网穿透配置 (tcp 模式, 无 token)
# ------------------------------------------
# 前提: 服务端 frps 配置里不能有 auth.token / [common] token, 否则会报
#       "token in login doesn't match token from configuration"
# 服务端改完记得: systemctl restart frps
# ==========================================
import os
import subprocess

FRP_HOST    = "usoren.usdream.dpdns.org"   # frps 所在主机
FRP_PORT    = 7000                         # frps bindPort
REMOTE_PORT = 8091                         # 公网暴露端口 (8090 已被占用, 改用 8091)
FRP_VER     = "0.56.0"
FRP_DIR     = f"/content/frp_{FRP_VER}_linux_amd64"
ACCESS_URL  = "http://" + FRP_HOST + ":" + str(REMOTE_PORT)

# --- 下载 frp 二进制 (几秒钟) ---
if not os.path.exists(f"{FRP_DIR}/frpc"):
    print("正在下载 frp ...")
    subprocess.run(
        "wget -qO- https://github.com/fatedier/frp/releases/download/"
        f"v{FRP_VER}/frp_{FRP_VER}_linux_amd64.tar.gz | tar -xz -C /content",
        shell=True)
assert os.path.exists(f"{FRP_DIR}/frpc"), "frpc 下载失败, 请重跑本格"
subprocess.run(f"chmod +x {FRP_DIR}/frpc", shell=True)

# --- 写 frpc.toml (注意: 没有 auth.token 这一行) ---
lines = [
    'serverAddr = "%s"' % FRP_HOST,
    'serverPort = %d' % FRP_PORT,
    'loginFailExit = false',
    'transport.tcpMux = true',
    'transport.poolCount = 5',
    'log.to = "/content/frpc.log"',
    'log.level = "info"',
    '',
    '[[proxies]]',
    'name = "comfyui_colab"',
    'type = "tcp"',
    'localIP = "127.0.0.1"',
    'localPort = 8188',
    'remotePort = %d' % REMOTE_PORT,
]
frpc_conf = "\n".join(lines) + "\n"

with open(f"{FRP_DIR}/frpc.toml", "w") as f:
    f.write(frpc_conf)

print("frpc.toml 已写入:")
print("-" * 50)
print(frpc_conf)
print("-" * 50)
print("启动后访问地址: " + ACCESS_URL)
print("地址必须带端口, 不带端口看到的是 frps 自带的 404 页")

In [ ]:
# ==========================================
# Cell 7: Google API Key 注入 —— 本工作流未使用, 整段已注释
# ------------------------------------------
# 原因: MiniMax-H3 的文本/视觉编码器是本地 Qwen3-VL-32B (CLIPLoader),
#       不调用任何云端 API。
# ==========================================

# import os
# from google.colab import userdata
# api_key = userdata.get('GOOGLE_API_KEY')
# os.environ["GOOGLE_API_KEY"] = api_key
# os.environ["GEMINI_API_KEY"] = api_key

print("已跳过 Google API 认证补丁 (本工作流不需要)")

In [ ]:
# ==========================================
# Cell 8: 启动 ComfyUI + FRP 穿透 (重启界面时只跑这一格)
# ------------------------------------------
# 依赖 Cell 6 已生成 frpc.toml
# ==========================================
import os
import time
import threading
import subprocess
import configparser

COMFY       = "/content/ComfyUI"
FRP_DIR     = "/content/frp_0.56.0_linux_amd64"
ACCESS_URL  = "http://usoren.usdream.dpdns.org:8091"

assert os.path.isdir(COMFY), "找不到 %s, 请先执行 Cell 1" % COMFY
assert os.path.exists(f"{FRP_DIR}/frpc.toml"), "找不到 frpc.toml, 请先跑 Cell 6"
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

# ---------------------------------------------------------
# 0. 防断开保活线程
# ---------------------------------------------------------
def keep_alive():
    while True:
        time.sleep(300)
        print("\n[Keep-Alive] 保持 Colab 连接活跃中...")

print("启动防断开后台保活线程...")
threading.Thread(target=keep_alive, daemon=True).start()

# ---------------------------------------------------------
# 1. 启动 frpc (秒级, 不用等 ComfyUI)
# ---------------------------------------------------------
for log in ["/content/comfy.log", "/content/frpc.log"]:
    if os.path.exists(log):
        os.remove(log)

print("正在后台唤起 FRP 穿透服务...")
subprocess.run(f"pkill -f '{FRP_DIR}/frpc' || true", shell=True)
subprocess.Popen(f"{FRP_DIR}/frpc -c {FRP_DIR}/frpc.toml "
                 ">> /content/frpc.log 2>&1", shell=True)

time.sleep(6)
frp_log = ""
if os.path.exists("/content/frpc.log"):
    frp_log = open("/content/frpc.log", errors="ignore").read()

if "start proxy success" in frp_log:
    print("FRP 隧道已建立 -> " + ACCESS_URL)
elif "token in login" in frp_log:
    print("服务端启用了 token 验证, 而客户端没有带")
    print("   请在 VPS 上删除 frps 配置里的 auth.token 后重启: systemctl restart frps")
elif "port already used" in frp_log or "already used" in frp_log:
    print("远程端口已被占用, 请把 Cell 6 的 REMOTE_PORT 改成另一个空闲端口")
elif "port not allowed" in frp_log:
    print("端口不在 frps 的 allowPorts 范围内, 请改端口或调服务端 allowPorts")
else:
    print("frpc 日志 (尚未看到 start proxy success):")
print(frp_log[-1200:] if frp_log else "(frpc 日志为空)")

# ---------------------------------------------------------
# 2. 拦截 ComfyUI-Manager 启动时的联网 Fetch (加快启动)
# ---------------------------------------------------------
print("\n配置 Manager 网络模式以跳过 Fetch...")
manager_config_paths = [
    f"{COMFY}/user/__manager/config.ini",
    f"{COMFY}/user/default/ComfyUI-Manager/config.ini",
    f"{COMFY}/custom_nodes/ComfyUI-Manager/config.ini",
]
for config_path in manager_config_paths:
    os.makedirs(os.path.dirname(config_path), exist_ok=True)
    config = configparser.ConfigParser()
    if os.path.exists(config_path):
        config.read(config_path)
    if "default" not in config:
        config["default"] = {}
    config["default"]["network_mode"] = "private"
    with open(config_path, "w") as f:
        config.write(f)
print("已切断 Fetch ComfyRegistry Data 流程")

# ---------------------------------------------------------
# 3. 启动 ComfyUI 主进程
#    96GB 显存: --highvram 让 32B 文本编码器与 DiT 常驻, 避免反复换入换出
# ---------------------------------------------------------
LAUNCH = ("python main.py --listen 127.0.0.1 --port 8188 "
          "--enable-cors-header '*' --preview-method auto "
          "--highvram --cache-lru 4")

print("\n正在启动 ComfyUI 主进程...")
subprocess.Popen(LAUNCH + " > /content/comfy.log 2>&1", shell=True, cwd=COMFY)

ready = False
for i in range(150):
    time.sleep(2)
    if not os.path.exists("/content/comfy.log"):
        continue
    txt = open("/content/comfy.log", errors="ignore").read()
    if "To see the GUI go to" in txt:
        ready = True
        print("ComfyUI ready")
        break
    if "Traceback" in txt and i > 10:
        print("启动报错, 日志尾部:")
        print(txt[-3000:])
        break

if not ready:
    print("--- 未就绪, 日志尾部 ---")
    if os.path.exists("/content/comfy.log"):
        print(open("/content/comfy.log", errors="ignore").read()[-3000:])

print("\n============================================================")
print("ComfyUI : " + ACCESS_URL)
print("工作流  : 左侧 Workflows -> video_minimax_h3_r2v")
print("地址必须带 :8091, 不带端口看到的是 frps 的 404 页")
print("============================================================\n")

# 保持本 Cell 不退出, 实时跟踪 ComfyUI 日志
subprocess.run("tail -f /content/comfy.log", shell=True)

## 工作流参数速查（对照 JSON widget 值）

| 节点 | 参数 | 模板值 | 96GB 下的建议 |
| --- | --- | --- | --- |
| MiniMaxH3ReferenceToVideo (136) | width × height | 1344 × 768 | 可上调到 1920×1088 |
| MiniMaxH3ReferenceToVideo | length | 124 帧 | 由 132 的秒数 × 24 自动算，公式在 131 |
| MiniMaxH3ReferenceToVideo | ref_image_size | match | 参考图细节要保留时改 `max` |
| PrimitiveFloat (132) | 时长 | 5 秒 | 96GB 可到 10–15 秒 |
| BasicScheduler (124) | scheduler / steps | simple / 20 | 参考图信息多时用 `beta` 或 `normal`，步数 25–30 |
| CreateVideo (130) | fps | 24 | 与 131 的公式一致，改了要同步 |
| ComfyMathExpression (131) | 帧数公式 | `max(5, round(a*24)) + (5 - (max(5, round(a*24)) % 17)) % 17` | 帧数需对齐 17 的倍数 +5，别手改 length |

## FRP 排错

| 日志关键字 | 原因 | 处理 |
| --- | --- | --- |
| `token in login doesn't match` | 服务端开了 token，客户端没带 | 删除 frps 的 `auth.token` 并 `systemctl restart frps`，或在 Cell 6 的 lines 里加 `auth.token = "xxx"` |
| `port already used` | 8091 被别的隧道占了 | 改 Cell 6 的 `REMOTE_PORT` |
| `port not allowed` | 不在 frps `allowPorts` 范围 | 改端口或调服务端 |
| 一直没有 `start proxy success` | frps 没起来 / 防火墙挡了 7000 | 在 VPS 上 `ss -lntp | grep 7000` 检查 |

> ComfyUI 走 WebSocket，用 tcp 类型隧道直连端口最省事；若改成 http 类型，Nginx 反代必须转发 `Upgrade` 和 `Connection` 头。